# 03 Fisher-Kolmogorov Calibration


In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import minimize


def graph_laplacian(W):
    W = np.asarray(W, dtype=float)
    return np.diag(W.sum(axis=1)) - W


def fisher_kolmogorov_rhs(t, y, L, kappa, alpha):
    return -kappa * (L @ y) + alpha * y * (1.0 - y)


def simulate_fk(times, y0, L, kappa, alpha):
    sol = solve_ivp(
        fisher_kolmogorov_rhs,
        t_span=(float(times[0]), float(times[-1])),
        y0=np.asarray(y0, dtype=float),
        t_eval=np.asarray(times, dtype=float),
        args=(L, float(kappa), float(alpha)),
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return np.clip(sol.y.T, 0.0, 1.0)


def fk_calibration_loss(theta, times, observed_tau, L):
    log_kappa, alpha = theta
    kappa = np.exp(log_kappa)
    predicted = simulate_fk(times, observed_tau[0], L, kappa, alpha)
    residual = predicted[1:] - observed_tau[1:]
    return float(np.nanmean(residual ** 2))


def fit_subject_fk_parameters(times, observed_tau, W):
    L = graph_laplacian(W)
    result = minimize(
        fk_calibration_loss,
        x0=np.array([np.log(0.05), 0.1]),
        args=(np.asarray(times, dtype=float), np.asarray(observed_tau, dtype=float), L),
        method="L-BFGS-B",
        bounds=[(np.log(1e-6), np.log(10.0)), (-2.0, 2.0)],
    )
    log_kappa, alpha = result.x
    return {
        "kappa": float(np.exp(log_kappa)),
        "alpha": float(alpha),
        "mse": float(result.fun),
        "optimizer_success": bool(result.success),
    }


def evaluate_fk_prediction(times, observed_tau, W, kappa, alpha):
    from scipy.stats import pearsonr

    predicted = simulate_fk(times, observed_tau[0], graph_laplacian(W), kappa, alpha)
    obs = np.asarray(observed_tau[1:], dtype=float).ravel()
    pred = np.asarray(predicted[1:], dtype=float).ravel()
    mask = np.isfinite(obs) & np.isfinite(pred)
    R, p = pearsonr(obs[mask], pred[mask])
    return {
        "mse": float(np.mean((pred[mask] - obs[mask]) ** 2)),
        "mae": float(np.mean(np.abs(pred[mask] - obs[mask]))),
        "R": float(R),
        "p_value": float(p),
        "n_values": int(mask.sum()),
    }
